In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from isabelle_connector.isabelle_connector import IsabelleConnector
from isabelle_connector.isabelle_types import Theory, TheoryConfig
from isabelle_connector.decorators import theory_builder

In [3]:
import re

def thy_from_text(theory_text: str, working_directory: str = "./isabelle_working_dir") -> Theory:
    """
    Build a Theory object from raw Isabelle theory text.
    Expects a header like: theory <TheoryName>
    """
    if not isinstance(theory_text, str) or not theory_text.strip():
        raise ValueError("Theory text must be a non-empty string")

    # Remove simple block comments to avoid matching a commented-out header.
    text_wo_comments = re.sub(r"\(\*.*?\*\)", "", theory_text, flags=re.DOTALL)
    match = re.search(
        r"^\s*theory\s+([A-Za-z][A-Za-z0-9_']*)\b",
        text_wo_comments,
        flags=re.MULTILINE,
    )
    if not match:
        raise ValueError("Could not find Isabelle theory header: expected `theory <Name>`")

    theory_name = match.group(1)

    body_match = re.search(
        r"\bbegin\b(.*)\bend\b\s*$",
        theory_text,
        flags=re.DOTALL,
    )
    queries = [body_match.group(1).strip()] if body_match and body_match.group(1).strip() else []

    return Theory(
        name=theory_name,
        working_directory=working_directory,
        queries=queries,
        is_temp=True,
    )

In [4]:
@theory_builder(prefix="Consts")
def consts_of_theory(src_theory: Theory, theory_config: TheoryConfig) -> str:
    """
    Create a temporary theory that extracts constants (name + type) from src_theory.
    """
    theory_config.imports += [f"{src_theory.working_directory}/{src_theory.name}"]
    return r"""
        let
            val {constants, ...} = Consts.dest (Sign.consts_of @{theory});
            val tconsts = map (fn (name, (typ, trm)) => (name, typ)) constants
        in
            tconsts
        end
    """


In [19]:
from isabelle_connector.config import INTERIM_DATA_DIR


isabelle = IsabelleConnector(
    name="extract_transitions", working_directory=str(INTERIM_DATA_DIR), debug=True
)

In [20]:
thy_text = """
theory Test
  imports Main
begin
ML\\<open> let val res = "Hello, World!" in res end \\<close>
end
"""

In [21]:
thy_object = thy_from_text(thy_text, working_directory=isabelle.working_directory)

In [22]:
results = isabelle.use_theories([thy_object], rm_if_temp=False)

Using cached results for 0 / 1 theories
Starting session 1 / 1: HOL


DONE:   0%|          | 0/1 [00:00<?, ?it/s]

Successful values from 1 / 1 theories
func:use_theories took: 6.900860548019409 sec


In [23]:
from isabelle_connector.config import INTERIM_DATA_DIR


consts_config = TheoryConfig(working_directory=str(INTERIM_DATA_DIR), session="HOL", imports=[])
consts_thy = consts_of_theory(thy_object, theory_config=consts_config)
results = isabelle.use_theories([consts_thy], rm_if_temp=False)

Using cached results for 0 / 1 theories


DONE:   0%|          | 0/1 [00:00<?, ?it/s]

Successful values from 1 / 1 theories
func:use_theories took: 2.8237123489379883 sec


In [25]:
results[consts_thy].values

[[('Record.tuple_isomorphism.typerep_tuple_isomorphism_IITN_tuple_isomorphism_inst.typerep_tuple_isomorphism_IITN_tuple_isomorphism',
   "(?'a, ?'b,        ?'c) tuple_isomorphism.tuple_isomorphism_IITN_tuple_isomorphism itself       \\<Rightarrow> typerep"),
  ('Quickcheck_Narrowing.narrowing_type.typerep_narrowing_type_IITN_narrowing_type_inst.typerep_narrowing_type_IITN_narrowing_type',
   'narrowing_type.narrowing_type_IITN_narrowing_type itself \\<Rightarrow> typerep'),
  ('Quickcheck_Narrowing.narrowing_term.typerep_narrowing_term_IITN_narrowing_term_inst.typerep_narrowing_term_IITN_narrowing_term',
   'narrowing_term.narrowing_term_IITN_narrowing_term itself \\<Rightarrow> typerep'),
  ('Quickcheck_Narrowing.narrowing_cons.typerep_narrowing_cons_IITN_narrowing_cons_inst.typerep_narrowing_cons_IITN_narrowing_cons',
   "?'a narrowing_cons.narrowing_cons_IITN_narrowing_cons itself       \\<Rightarrow> typerep"),
  ('Record.tuple_isomorphism.typerep_tuple_isomorphism_pre_tuple_isomor